# ATINA — gerar imagens de soluções com IA

Este notebook executa o gerador do ATINA no Google Colab. Usa GPU quando disponível, guarda imagens no Google Drive e envia-as para o Supabase. **Não partilhes este notebook com chaves ou CSVs privados.**

A utilização gratuita do Colab é limitada e a GPU pode não estar sempre disponível. Executa em lotes e retoma o mesmo comando se a sessão terminar.

## 1. Preparar os ficheiros privados

Como o carregamento direto de ficheiros pode falhar em alguns telemóveis, primeiro abre a app Google Drive e carrega o CSV exportado do Supabase para **O meu disco** (My Drive). Mantém o ficheiro privado. Depois executa a célula de código abaixo: ela liga o Drive, encontra o CSV e prepara a pasta de imagens.

O CSV contém respostas e pistas; não o partilhes nem o tornes público.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

# Primeiro, carrega o CSV na tua pasta Google Drive usando a app Google Drive.
# Depois executa esta célula e indica o nome exato do ficheiro, se necessário.
drive.mount('/content/drive')
drive_root = Path('/content/drive/MyDrive')
csv_candidates = list(drive_root.rglob('*.csv'))
print('Ficheiros CSV encontrados:')
for i, path in enumerate(csv_candidates):
    print(f'{i}: {path.relative_to(drive_root)}')
assert csv_candidates, 'Não encontrei CSV no Google Drive. Carrega primeiro o ficheiro para o Drive e volta a executar.'
if len(csv_candidates) == 1:
    selected_csv = csv_candidates[0]
else:
    choice = int(input('Escreve o número do CSV challenges exportado do Supabase: '))
    selected_csv = csv_candidates[choice]
CSV_PATH = '/content/challenges.csv'
shutil.copy2(selected_csv, CSV_PATH)
OUTPUT_DIR = '/content/drive/MyDrive/ATINA-generated/images'
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print('CSV preparado:', selected_csv.name)


## 2. Instalar o gerador

O código é obtido da branch de preparação do ATINA. A instalação e a primeira descarga do modelo podem demorar bastante.

In [ ]:
!git clone --depth 1 --branch feature/ai-solution-images https://github.com/lufime/ATINA.git /content/ATINA
%cd /content/ATINA/tools/ai-solution-images
!pip -q install -r requirements.txt
import torch
print('CUDA disponível:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Ativa uma GPU em Ambiente de execução > Alterar tipo de ambiente de execução > T4 GPU, se estiver disponível.')


## 3. Guardar as credenciais com segurança

No Colab, abre o painel **Secrets** (ícone de chave) e cria `SUPABASE_URL` e `SUPABASE_SERVICE_ROLE_KEY`. Ativa o acesso destes secrets para este notebook. Usa a chave service_role apenas nos Secrets do Colab; nunca a escrevas no código, CSV ou GitHub.

In [ ]:
from google.colab import userdata
import os
os.environ['SUPABASE_URL'] = userdata.get('SUPABASE_URL')
os.environ['SUPABASE_SERVICE_ROLE_KEY'] = userdata.get('SUPABASE_SERVICE_ROLE_KEY')
assert os.environ['SUPABASE_URL'] and os.environ['SUPABASE_SERVICE_ROLE_KEY']
print('Credenciais carregadas sem as mostrar.')


## 4. Testar primeiro com 5 imagens

Confirma a qualidade antes de avançar. A geração pode demorar e a GPU gratuita não é garantida.

In [ ]:
!python generate.py --manifest "$CSV_PATH" --output "$OUTPUT_DIR" --limit 5 --upload


## 5. Gerar o conjunto completo

Quando estiveres satisfeito com o teste, executa a célula seguinte. Se a sessão terminar, volta a executar o notebook desde o início e repete: as imagens já existentes no Google Drive são reutilizadas e o script volta a tentar o envio para o Storage. O processo pode precisar de várias sessões.

In [ ]:
!python generate.py --manifest "$CSV_PATH" --output "$OUTPUT_DIR" --upload


## 6. Guardar o SQL de atualização

O gerador cria SQL para registar os ativos como `generated` e atualizar `solution_image_url`. Revê as imagens antes de aprovar. Guarda o SQL no Drive; não o executes sem revisão.

In [ ]:
from pathlib import Path
sql_file = Path('/content/ATINA/tools/ai-solution-images/generated/apply-generated-images.sql')
if sql_file.exists():
    destination = Path('/content/drive/MyDrive/ATINA-generated/apply-generated-images.sql')
    shutil.copy2(sql_file, destination)
    print('SQL guardado no Google Drive:', destination)
else:
    print('Ainda não foi criado SQL. Confirma se pelo menos uma imagem foi enviada.')
